# CGC Engine Native Capability Test - Google Colab

這個筆記本用於在 Google Colab 免費 GPU（T4 16GB）上測試模型的原生能力，**沒有 expert cache、MTP、自定義 chat template 等修改**，作為基線對比。

## 測試目標
1. 比較 denseIQ4X（我們的量化）vs Q4_K_M（標準量化）的品質和速度
2. 在沒有內存壓力限制（T4 16GB GPU）下，測試模型的真實反應
3. 對比本地 Mac 16GB（有內存壓力）vs Colab T4 16GB（無內存壓力）的表現

## 模型來源
- Hugging Face: `Alexchuang/cgcengine-models`
- denseIQ4X: `Nail-Qwen3.6-35B-A3B-MTP-UD-IQ3_XXS-denseIQ4X.gguf`
- Q4_K_M: `Nail-Qwen3.6-35B-A3B-UD-Q4_K_M.gguf`（如果存在）


## 步驟 1: 檢查 GPU 環境

In [ ]:
!nvidia-smi
!df -h /content
!free -h

## 步驟 2: 安裝依賴和 llama.cpp

In [ ]:
# 安裝 huggingface_hub
!pip install -q huggingface_hub[cli] requests gradio

# 克隆並構建 llama.cpp（upstream 原版，沒有我們的修改）
!git clone https://github.com/ggerganov/llama.cpp.git
%cd llama.cpp
!cmake -B build -DGGML_CUDA=on -DCMAKE_BUILD_TYPE=Release
!cmake --build build --config Release -j$(nproc)
!ls -lh build/bin/llama-server

## 步驟 3: 從 Hugging Face 下載模型

In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'  # 使用鏡像加速下載

!mkdir -p /content/models

# 下載 denseIQ4X 模型（我們的量化）
print('下載 denseIQ4X 模型...')
!hf download Alexchuang/cgcengine-models Nail-Qwen3.6-35B-A3B-MTP-UD-IQ3_XXS-denseIQ4X.gguf --local-dir /content/models

# 嘗試下載 Q4_K_M 模型（標準量化，如果存在）
print('嘗試下載 Q4_K_M 模型...')
!hf download Alexchuang/cgcengine-models Nail-Qwen3.6-35B-A3B-UD-Q4_K_M.gguf --local-dir /content/models 2>&1 || echo 'Q4_K_M 模型不存在，只測試 denseIQ4X'

!ls -lh /content/models/

## 步驟 4: 定義測試用例和對比函數

In [ ]:
import requests
import json
import time
import subprocess

# 測試用例
TEST_CASES = [
    ("T1: Short EN Q&A", "2+2 is? Answer with one word."),
    ("T2: Short CN Q&A", "15+27等於多少？只回答數字"),
    ("T3: Code Gen", "Write a Python function to calculate fibonacci."),
    ("T4: Logic", "If all cats are animals and all animals are living things, are all cats living things? Answer yes or no."),
    ("T5: Long Context", "請用一段中文說明為什麼巴黎會成為法國的政治與文化中心，包括歷史、地理和文化三個方面。"),
]

def start_server(model_path, port):
    """啟動 llama-server"""
    cmd = [
        '/content/llama.cpp/build/bin/llama-server',
        '-m', model_path,
        '-ngl', '99',
        '-c', '8192',
        '--host', '127.0.0.1',
        '--port', str(port),
        '-t', '8',
        '--temp', '0.7',
        '--top-p', '0.9',
        '--repeat-penalty', '1.1'
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(15)  # 等待 server 啟動
    return proc

def test_api(port, name, prompt, temperature=0.7, max_tokens=100):
    """測試 API"""
    try:
        data = {
            'model': 'default',
            'messages': [{'role': 'user', 'content': prompt}],
            'temperature': temperature,
            'max_tokens': max_tokens
        }
        start = time.time()
        response = requests.post(f'http://127.0.0.1:{port}/v1/chat/completions', json=data, timeout=120)
        elapsed = time.time() - start
        result = response.json()
        
        content = result['choices'][0]['message']['content']
        timings = result.get('timings', {})
        decode = timings.get('predicted_per_second', 0)
        prompt_tps = timings.get('prompt_per_second', 0)
        
        return {
            'name': name,
            'content': content,
            'decode': decode,
            'prompt': prompt_tps,
            'elapsed': elapsed,
            'error': None
        }
    except Exception as e:
        return {'name': name, 'content': '', 'decode': 0, 'prompt': 0, 'elapsed': 0, 'error': str(e)}

def run_all_tests(port, label):
    """運行所有測試用例"""
    print(f'\n{"="*80}')
    print(f'運行測試: {label}')
    print(f'{"="*80}')
    
    results = []
    for name, prompt in TEST_CASES:
        print(f'\n測試: {name}')
        result = test_api(port, name, prompt)
        status = '✅ PASS' if result['content'] and len(result['content']) > 10 else '❌ FAIL'
        print(f'  {status} | Decode: {result["decode"]:.2f} t/s | Prefill: {result["prompt"]:.2f} t/s')
        print(f'  Content: {repr(result["content"][:150])}')
        results.append(result)
    
    return results

print('測試函數已定義完成')

## 步驟 5: 啟動 denseIQ4X 模型並測試

In [ ]:
# 啟動 denseIQ4X 模型（端口 8080）
denseiq4x_path = '/content/models/Nail-Qwen3.6-35B-A3B-MTP-UD-IQ3_XXS-denseIQ4X.gguf'
print(f'啟動 denseIQ4X 模型: {denseiq4x_path}')
proc_dense = start_server(denseiq4x_path, 8080)
print('denseIQ4X server 已啟動 (端口 8080)')

# 運行測試
results_dense = run_all_tests(8080, 'denseIQ4X (我們的量化)')

## 步驟 6: 啟動 Q4_K_M 模型並測試（如果存在）

In [ ]:
import os

q4km_path = '/content/models/Nail-Qwen3.6-35B-A3B-UD-Q4_K_M.gguf'

if os.path.exists(q4km_path):
    # 先停止 denseIQ4X server（釋放 GPU 內存）
    proc_dense.terminate()
    time.sleep(5)
    
    # 啟動 Q4_K_M 模型（端口 8081）
    print(f'啟動 Q4_K_M 模型: {q4km_path}')
    proc_q4km = start_server(q4km_path, 8081)
    print('Q4_K_M server 已啟動 (端口 8081)')
    
    # 運行測試
    results_q4km = run_all_tests(8081, 'Q4_K_M (標準量化)')
else:
    print('Q4_K_M 模型不存在，跳過對比測試')
    results_q4km = None

## 步驟 7: 對比結果

In [ ]:
import pandas as pd

# 構建對比表格
comparison_data = []
for i, (name, _) in enumerate(TEST_CASES):
    row = {'測試用例': name}
    
    if results_dense and i < len(results_dense):
        row['denseIQ4X Decode'] = f'{results_dense[i]["decode"]:.2f} t/s'
        row['denseIQ4X 長度'] = len(results_dense[i]['content'])
        row['denseIQ4X 內容'] = results_dense[i]['content'][:80]
    
    if results_q4km and i < len(results_q4km):
        row['Q4_K_M Decode'] = f'{results_q4km[i]["decode"]:.2f} t/s'
        row['Q4_K_M 長度'] = len(results_q4km[i]['content'])
        row['Q4_K_M 內容'] = results_q4km[i]['content'][:80]
    
    comparison_data.append(row)

df = pd.DataFrame(comparison_data)
print('='*100)
print('對比結果總結')
print('='*100)
print(df.to_string(index=False))

# 保存結果
df.to_csv('/content/comparison_results.csv', index=False, encoding='utf-8-sig')
print('\n結果已保存到 /content/comparison_results.csv')

## 步驟 8: 總結與分析

In [ ]:
print('='*80)
print('測試總結與分析')
print('='*80)

if results_dense:
    avg_decode_dense = sum(r['decode'] for r in results_dense) / len(results_dense)
    avg_len_dense = sum(len(r['content']) for r in results_dense) / len(results_dense)
    print(f'\ndenseIQ4X（我們的量化）:')
    print(f'  平均 Decode: {avg_decode_dense:.2f} t/s')
    print(f'  平均輸出長度: {avg_len_dense:.0f} chars')

if results_q4km:
    avg_decode_q4km = sum(r['decode'] for r in results_q4km) / len(results_q4km)
    avg_len_q4km = sum(len(r['content']) for r in results_q4km) / len(results_q4km)
    print(f'\nQ4_K_M（標準量化）:')
    print(f'  平均 Decode: {avg_decode_q4km:.2f} t/s')
    print(f'  平均輸出長度: {avg_len_q4km:.0f} chars')

print(f'\n{'='*80}')
print('關鍵發現')
print(f'{'='*80}')
print('1. 在 Colab T4 16GB GPU 上，沒有內存壓力限制')
print('2. 可以比較 denseIQ4X vs Q4_K_M 兩種量化的品質和速度')
print('3. 對比本地 Mac 16GB（有內存壓力）的表現')
print('4. 確定問題是模型本身的限制，還是我們的配置問題')

## 下載結果

測試完成後，可以從左側文件面板下載 `comparison_results.csv`。